In [1]:
# Custom
import sys
sys.path.insert(0, '../src')
from args import args
import utils


## Utils and Inference
import csv
from collections import Counter
import json
import numpy as np
import pandas as pd
import re

import os
from tqdm import tqdm

/u/mjkishan/miniconda3/envs/syn_data/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.0.post2)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


In [2]:
import sys
sys.path.insert(0, '..')
from prompts import icl_source_code_prompts

In [ ]:
class args(args):

    setting = ["batching", "single_variable"][1]

    experimental_setting = ["vanilla", "full_source_code", "source_code_grep_chunk", "paragraph_code"][1]
    
    setting = ["batching", "single_variable"][1]
    saving_path = f"../saved_data/inference/{setting}/{experimental_setting}"
    
    variable_column = "variable_name"
    program_path_column = "corrected_path"
    
    data_path  = '../data/get_variable_para_info_formatted_info_annotation_set_23_06_2026_with_path.json'

    platform = ["llm_provider_1", "watsonx", "mse", "litellm", "azure"][3]



In [20]:
## Check if the saving directories exist or not; If not, then create them
utils.create_save_directories()

Base directory already exists: saved_data
Subdirectory already exists: ./saved_data/evaluation_results/
Subdirectory already exists: ./saved_data/inference/


In [21]:
### Check if the experiments saving directories exist or not; If not, then create them
utils.ensure_exp_dirs_exist(args.saving_path)

Directory already exists: ./saved_data/inference/single_variable/full_source_code


'../saved_data/inference/single_variable/full_source_code'

In [ ]:
prompt = icl_source_code_prompts.source_code_prompts_v3_detailed
print(prompt)

You are a COBOL and software programming expert. For a given variable, generate the following (in English):
1. Short Description: Expand the variable name by replacing each abbreviation with its full form.
2. Long Description: A concise phrase using the expanded forms that captures the variable's purpose, business semantics, and interaction-driven implications, by considering all occurrences of the variable in the source code, and remaining strictly faithful to the code.

COBOL-Specific Expansions (prioritize when applicable)
WS = Working-Storage
DFH = CICS
FD = File Description
RESP = Response Code
CA = Communication Area
EIB = Execute Interface Block

Output Format
[
  {
    "input": "provide input variable",
    "short description": "expanded form of the given input as a string in English",
    "long description": "concise phrase (strictly in English) aggregated from all variable occurrences, capturing its purpose, business semantics, and interaction-driven implications, while remai

## Search Source Code

In [23]:

def read_file_binary_cobol(full_path):
    
    try:
        with open(full_path, "rb") as f:
            content = f.read()

        return {
            "path": full_path,
            "content": content.decode("latin-1")  # bytes → string
        }

    except Exception as e:
        return {
            "error": f"Found file at {full_path} but could not read it: {e}"
        }

    return {"error": "File not found."}



## Gather and Merge Paragraph Context

In [24]:
def gather_context(corrected_paths):
    """
    Iterate over all paths in corrected_paths, read each file,
    and collate them with [start of file_name] [end of file_name] markers.
    
    Args:
        corrected_paths: List of file paths to read
    Returns:
        List of source code strings with file markers

    Note that the deduplication is implemented in during the inference call
    """
    import os
    
    source_codes = []
    
    # Iterate over all paths in corrected_paths
    for file_path in corrected_paths:
        # Read the file
        file_content = read_file_binary_cobol(full_path=file_path)
        
        if 'error' in file_content:
            # Skip files that couldn't be read
            continue
        
        # Get the file name from the path
        file_name = os.path.basename(file_path)
        
        # Get the source code content
        source_code = file_content['content']
        
        
        # Add file markers
        marked_content = f"[start of {file_name}]\n{source_code}\n[end of {file_name}]"
        source_codes.append(marked_content)
    
    return source_codes

def merge_context(source_codes):
    merged_context = "\n\n".join(source_codes)
    return merged_context


### Data Preprocessing

In [25]:
data_df = pd.read_json(args.data_path)

In [27]:
variable_names = list(data_df[args.variable_column])

### Validating and Loading the Inference Pipeline

In [30]:
inference = utils.Inference(args)

In [ ]:
model_names = utils.validate_api_endpoints(args, args.platform)

print(f"Models from {args.platform}: {model_names}\n" )

print(f"Selected Models from {args.platform}: {model_names}" )

API Endpoints Validated successfully!
Models from litellm: ['claude_sonnet_4_5']

Selected Models from litellm: ['claude_sonnet_4_5']


In [33]:
pred_folder=args.saving_path+"/"
pred_folder

'../saved_data/inference/single_variable/full_source_code/'

In [34]:

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)

def write_json_record(file_path: str, record: dict) -> None:
    """Append a record to a JSON array file."""
    with open(file_path, 'r+') as jsonfile:
        data = json.load(jsonfile)
        data.append(record)
        jsonfile.seek(0)
        json.dump(data, jsonfile, indent=2, cls=NumpyEncoder)
        jsonfile.truncate()

In [ ]:
file_names = []

for model_name in tqdm(model_names, desc="Models", position=0):
    
    prediction_df = data_df
    file_name = f"{model_name}_variable_DD_{args.experimental_setting}_.json"
    
    file_name = utils.update_file_name(file_name, pred_folder=pred_folder, extension=".json")

    
    # Checkpointing Logic- model_name can be a string representing model identifier. 
    # The model_name and file_name are intended to be manually defined to resume checkpointing. This assume that the file is already created.
    
    if model_name == '':
        file_name = ''
        checkpoint_index = 10
        
    else:
        checkpoint_index = 0
        with open(pred_folder + file_name, 'w') as jsonfile:
            json.dump([], jsonfile)

    
    print(f"Starting Inference for {model_name} in {file_name}")
    pbar = tqdm(total=len(variable_names), desc=f"Inference [{model_name}]",
                position=1, leave=False, dynamic_ncols=True)
    

    full_path = pred_folder + file_name
    
    fields = list(prediction_df.columns) + ["prompt_used", "token_statistics", "predictions_DD"]

    for i, var_name in enumerate(variable_names):
        if i < checkpoint_index:
            pbar.update(1)
            continue

        if i == checkpoint_index:
            print(f"starting inference from index {i}")

        row_data = list(prediction_df.iloc[i])
        current_prompt = prompt[:]

        # Ensure Deduplication to avoid repetitions in the prompt with same program
        current_program_paths = list(set(prediction_df.iloc[i][args.program_path_column]))


        current_prompt = current_prompt.replace("{{TEST_VARIABLE}}", f"{var_name}")

        try:
            code_chunks = gather_context(corrected_paths=current_program_paths)

            if len(code_chunks) == 0:
                err_string = "error found: No files could be read"
                record = dict(zip(fields, row_data + ["", err_string, err_string]))
                
                write_json_record(full_path, record)
                pbar.update(1)
                continue

            source_code_string = merge_context(code_chunks)

        except Exception as e:
            err_string = f"error found: {e}"
            record = dict(zip(fields, row_data + ["", err_string, err_string]))
            write_json_record(full_path, record)
            pbar.update(1)
            continue

        try:
            current_prompt = current_prompt.replace("{{SOURCE_CONTEXT}}", f"{source_code_string}")
            
            response, raw_output = inference(prompt=current_prompt, inference_platform=args.platform,
                                             model_name=model_name, return_raw=True)
            token_stats = dict(dict(raw_output)['usage'])

            # Filtering For serialization
            token_stats = {k: token_stats[k] for k in ('completion_tokens', 'prompt_tokens', 'total_tokens')}
            
        except Exception as e:
            if not response and not token_stats:
                response = token_stats = f"error found: {e}"

            else: 
                response = str(response)
                token_stats = str(response)

        record = dict(zip(fields, row_data + [current_prompt, token_stats, response]))
        write_json_record(full_path, record)
        pbar.update(1)

    pbar.close()
    file_names.append(file_name)